## DATASET DOWNLOAD

In [ ]:
!mkdir -p downloads
!cd downloads

In [ ]:
import pandas as pd
import json

# 1. Download the dataset directly to Colab (High bandwidth)
# Note: You'll need your kaggle.json uploaded to Colab for this
!kaggle datasets download -d yelp-dataset/yelp-dataset
!unzip yelp-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset
License(s): other
100% 4.07G/4.07G [00:35<00:00, 122MB/s]

Archive:  yelp-dataset.zip
  inflating: Dataset_User_Agreement.pdf  
  inflating: yelp_academic_dataset_business.json  
  inflating: yelp_academic_dataset_checkin.json  
  inflating: yelp_academic_dataset_review.json  
  inflating: yelp_academic_dataset_tip.json  
  inflating: yelp_academic_dataset_user.json  


## EDA

In [ ]:
import pandas as pd
import json

# Loading a chunk of the dataset to check for replicas
# The full file is very large, so we'll inspect the first 100,000 reviews
reviews = []
with open('yelp_academic_dataset_review.json', 'r') as f:
    for i, line in enumerate(f):
        if i >= 100000: break
        reviews.append(json.loads(line))

df_reviews = pd.DataFrame(reviews)

# Check for duplicate user_ids and business_ids
user_counts = df_reviews['user_id'].value_counts()
business_counts = df_reviews['business_id'].value_counts()

print(f"Total reviews sampled: {len(df_reviews)}")
print(f"Unique users: {df_reviews['user_id'].nunique()}")
print(f"Unique businesses (products): {df_reviews['business_id'].nunique()}")

# Identify if replicas exist
print(f"\nUsers with multiple reviews: {(user_counts > 1).sum()}")
print(f"Businesses with multiple reviews: {(business_counts > 1).sum()}")

# Display top replicas
print("\nTop 5 Users by review count:")
display(user_counts.head())

print("\nTop 5 Businesses by review count:")
display(business_counts.head())


Total reviews sampled: 100000
Unique users: 79345
Unique businesses (products): 9973

Users with multiple reviews: 11666
Businesses with multiple reviews: 7520

Top 5 Users by review count:


,count
user_id,
_BcWyKQL16ndpBdggh2kNA,65
Xw7ZjaGfr0WNVt6s_5KZfA,38
1HM81n6n4iPIFU5d2Lokhw,31
0Igx-a1wAstiBDerGxXk2A,29
Um5bfs5DH6eizgjH3xZsvg,27



Top 5 Businesses by review count:


,count
business_id,
GBTPC53ZrG1ZBY3DT8Mbcw,950
PY9GRfzr4nTZeINf346QOw,460
W4ZEKkva9HpAdZG88juwyQ,433
vN6v8m4DO45Z4pp8yxxF_w,404
pSmOH4a3HNNpYM82J5ycLA,384


In [ ]:
import json

# Count total rows and get column names
count = 0
columns = []

with open('yelp_academic_dataset_review.json', 'r') as f:
    for line in f:
        if count == 0:
            # Get columns from the first line
            columns = list(json.loads(line).keys())
        count += 1

print(f'Total rows in yelp_academic_dataset_review.json: {count:,}')
print(f'Columns: {columns}')

Total rows in yelp_academic_dataset_review.json: 6,990,280
Columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']


In [ ]:
import json

def get_metadata_info(filename):
    with open(filename, 'r') as f:
        first_line = f.readline()
        return list(json.loads(first_line).keys())

business_cols = get_metadata_info('yelp_academic_dataset_business.json')
user_cols = get_metadata_info('yelp_academic_dataset_user.json')


print(f'Business Metadata Columns: {business_cols}')
print(f'\nUser Metadata Columns: {user_cols}')


Business Metadata Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

User Metadata Columns: ['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny', 'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot', 'compliment_more', 'compliment_profile', 'compliment_cute', 'compliment_list', 'compliment_note', 'compliment_plain', 'compliment_cool', 'compliment_funny', 'compliment_writer', 'compliment_photos']


In [ ]:
import json

# Let's look at one example of a business (product) metadata entry
with open('yelp_academic_dataset_business.json', 'r') as f:
    first_business = json.loads(f.readline())

import pprint
pprint.pprint(first_business)

{'address': '1616 Chapala St, Ste 2',
 'attributes': {'ByAppointmentOnly': 'True'},
 'business_id': 'Pns2l4eNsfO8kk83dixA6A',
 'categories': 'Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, '
               'Acupuncture, Health & Medical, Nutritionists',
 'city': 'Santa Barbara',
 'hours': None,
 'is_open': 0,
 'latitude': 34.4266787,
 'longitude': -119.7111968,
 'name': 'Abby Rappoport, LAC, CMQ',
 'postal_code': '93101',
 'review_count': 7,
 'stars': 5.0,
 'state': 'CA'}


## TRAIN DATA

In [ ]:
import pandas as pd
import json
import random
import os

# Set seed for reproducibility
random.seed(42)

def load_random_sample(filename, limit):
    print(f"Sampling {limit} lines from {filename}...")
    with open(filename, 'r') as f:
        lines = f.readlines()
    sampled = random.sample(lines, min(limit, len(lines)))
    return pd.DataFrame([json.loads(l) for l in sampled])

def get_elite_count(elite_val):
    if not elite_val or str(elite_val).lower() == 'none':
        return 0
    if isinstance(elite_val, str):
        return len([year for year in elite_val.split(',') if year.strip()])
    return 0

def flatten_attr(attr):
    if isinstance(attr, dict):
        return ', '.join([f'{k}: {v}' for k, v in attr.items()])
    return 'None'

# 1. Load data with random sampling
df_reviews = load_random_sample('yelp_academic_dataset_review.json', 500000)
df_users = load_random_sample('yelp_academic_dataset_user.json', 300000)
df_biz = load_random_sample('yelp_academic_dataset_business.json', 100000)

# 2. Preprocess
df_users['user_elite_count'] = df_users['elite'].apply(get_elite_count)
df_biz['biz_attributes_clean'] = df_biz['attributes'].apply(flatten_attr)

# 3. Merge
merged_df = pd.merge(df_reviews, df_users, on='user_id', how='inner', suffixes=('_rev', '_user'))
merged_df = merged_df.rename(columns={'stars': 'stars_review', 'name': 'user_name', 'review_count': 'user_review_count', 'fans': 'user_fans'})

final_df = pd.merge(merged_df, df_biz, on='business_id', how='inner', suffixes=('', '_biz'))
final_df = final_df.rename(columns={'name': 'biz_name', 'stars': 'biz_stars', 'review_count': 'biz_review_count', 'city': 'biz_city', 'state': 'biz_state'})

# 4. Filter for prolific users (>= 10 reviews in our sample)
user_counts = final_df['user_id'].value_counts()
prolific_users = user_counts[user_counts >= 7].index
final_dataset = final_df[final_df['user_id'].isin(prolific_users)].copy()

# 5. Per-user holdout split
final_dataset['date'] = pd.to_datetime(final_dataset['date'])
final_dataset = final_dataset.sort_values(['user_id', 'date'])

# Hold out the most recent review
test_rows = final_dataset.groupby('user_id').tail(1)
train_rows = final_dataset.drop(test_rows.index)

# 6. Target user count (limit to 1000 users in test set)
if test_rows['user_id'].nunique() > 1000:
    selected_users = random.sample(list(test_rows['user_id'].unique()), 1000)
    test_rows = test_rows[test_rows['user_id'].isin(selected_users)]
    train_rows = train_rows[train_rows['user_id'].isin(selected_users)]

# 7. Final Golden Columns (Fixed missing comma)
golden_columns = [
    'user_id', 'user_name', 'user_review_count', 'average_stars',
    'user_elite_count', 'user_fans', 'business_id', 'biz_name',
    'biz_stars', 'biz_city', 'biz_state', 'categories', 'biz_attributes_clean',
    'stars_review', 'text', 'date'
]

train_dataset = train_rows[golden_columns]
test_dataset = test_rows[golden_columns]

Sampling 500000 lines from yelp_academic_dataset_review.json...
Sampling 300000 lines from yelp_academic_dataset_user.json...
Sampling 100000 lines from yelp_academic_dataset_business.json...


In [ ]:
# Summary

print(f"\nTrain rows : {len(train_rows)}")
print(f"Avg train reviews per user: {len(train_rows) / test_rows['user_id'].nunique():.1f}")
print(f"Verification - Users with 0 train reviews: {(~test_rows['user_id'].isin(train_rows['user_id'])).sum()}")

print(f"Test rows  : {len(test_dataset)} ({test_dataset['user_id'].nunique()} unique users)")


Train rows : 4569
Avg train reviews per user: 10.7
Verification - Users with 0 train reviews: 0
Test rows  : 429 (429 unique users)


In [ ]:
display(train_dataset.head(1))

,user_id,user_name,user_review_count,average_stars,user_elite_count,user_fans,business_id,biz_name,biz_stars,biz_city,biz_state,categories,biz_attributes_clean,stars_review,text,date
12749,-B-QEUESGWHPE_889WJaeg,Mark,1014,3.41,12,200,-kfHz1kimX62QEceFkkB_g,Edomasa,3.5,Santa Barbara,CA,"Restaurants, Sushi Bars, Japanese","Alcohol: u'beer_and_wine', Corkage: False, Bus...",4.0,Cheap and easy. That's what you get with Edoma...,2010-02-21 00:06:29


In [ ]:
# Export the train dataset to a CSV file
train_output_filename = 'train.csv'
train_dataset.to_csv(train_output_filename, index=False)
print(f'Successfully exported {len(train_dataset)} rows to {train_output_filename}')

Successfully exported 4569 rows to train.csv


In [ ]:
from google.colab import files

# Download the file to your local machine
files.download('train.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## TEST DATA

In [ ]:
# display(test_dataset.head(1))

In [ ]:
# Export the test dataset to a CSV file
test_output_filename = 'test.csv'
test_dataset.to_csv(test_output_filename, index=False)
print(f'Successfully exported {len(test_dataset)} test rows to {test_output_filename}')

Successfully exported 429 test rows to test.csv


In [ ]:
from google.colab import files

# Download the file to your local machine
files.download("test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>